# Multivariate Time-Series Forecasting

## 1. Feature Engineering

This notebook creates forecasting features from the preprocessed retail sales dataset.

The objective is to transform the preprocessed time-series data into a feature-rich dataset suitable for baseline and deep learning forecasting models.

The feature engineering pipeline will include:

- Time-based features
- Lagged sales features
- Rolling statistical features
- Price-related features
- Event and SNAP-related features
- Product, store, department, category, and state information

All features will be constructed carefully to prevent future data leakage.

The preprocessed dataset will remain unchanged, and all engineered features will be stored separately.

In [1]:
import gc
import numpy as np
import pandas as pd
import pyarrow.parquet as pq

print("Libraries imported successfully.")

Libraries imported successfully.


In [2]:
# Path to the preprocessed dataset
data_path = "preprocessed_train.parquet"

# Open Parquet file without loading the full dataset into memory
parquet_file = pq.ParquetFile(data_path)

print("====================================")
print("Preprocessed Dataset Information")
print("====================================")

print("\nNumber of rows:")
print(f"{parquet_file.metadata.num_rows:,}")

print("\nNumber of columns:")
print(len(parquet_file.schema.names))

print("\nColumns:")
print(parquet_file.schema.names)

print("\nNumber of row groups:")
print(parquet_file.num_row_groups)

Preprocessed Dataset Information

Number of rows:
58,327,370

Number of columns:
23

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available']

Number of row groups:
61


In [3]:
# Read only one row group for feature inspection
sample = parquet_file.read_row_group(
    0,
    columns=[
        "d",
        "date",
        "weekday",
        "wday",
        "month",
        "year"
    ]
).to_pandas()

print("====================================")
print("Existing Time Features")
print("====================================")

print("\nColumns:")
print(sample.columns.tolist())

print("\nFirst 10 rows:")
print(sample.head(10))

print("\nUnique weekday values:")
print(sample["weekday"].unique())

print("\nUnique wday values:")
print(sorted(sample["wday"].unique()))

print("\nUnique months:")
print(sorted(sample["month"].unique()))

print("\nUnique years:")
print(sorted(sample["year"].unique()))

Existing Time Features

Columns:
['d', 'date', 'weekday', 'wday', 'month', 'year']

First 10 rows:
     d       date   weekday  wday  month  year
0  d_1 2011-01-29  Saturday     1      1  2011
1  d_1 2011-01-29  Saturday     1      1  2011
2  d_1 2011-01-29  Saturday     1      1  2011
3  d_1 2011-01-29  Saturday     1      1  2011
4  d_1 2011-01-29  Saturday     1      1  2011
5  d_1 2011-01-29  Saturday     1      1  2011
6  d_1 2011-01-29  Saturday     1      1  2011
7  d_1 2011-01-29  Saturday     1      1  2011
8  d_1 2011-01-29  Saturday     1      1  2011
9  d_1 2011-01-29  Saturday     1      1  2011

Unique weekday values:
['Saturday' 'Sunday' 'Monday' 'Tuesday' 'Wednesday' 'Thursday' 'Friday']

Unique wday values:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7)]

Unique months:
[np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12)

In [4]:
import pyarrow as pa
import pyarrow.parquet as pq
import gc
import os

input_path = "preprocessed_train.parquet"
output_path = "features_time.parquet"

input_file = pq.ParquetFile(input_path)

writer = None
total_rows = 0

for group_num in range(input_file.num_row_groups):

    df = input_file.read_row_group(group_num).to_pandas()

    # Ensure date is datetime
    df["date"] = pd.to_datetime(df["date"])

    # -----------------------------
    # Time-based features
    # -----------------------------

    df["day_of_month"] = df["date"].dt.day.astype("int8")

    df["week_of_year"] = (
        df["date"].dt.isocalendar().week.astype("int8")
    )

    df["day_of_year"] = (
        df["date"].dt.dayofyear.astype("int16")
    )

    df["quarter"] = (
        df["date"].dt.quarter.astype("int8")
    )

    df["is_weekend"] = (
        df["date"].dt.dayofweek >= 5
    ).astype("int8")

    total_rows += len(df)

    # Convert to Parquet table
    table = pa.Table.from_pandas(
        df,
        preserve_index=False
    )

    # Create writer using first group's schema
    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)

    print(
        f"Row group {group_num + 1}/{input_file.num_row_groups} processed"
    )

    del df, table
    gc.collect()

writer.close()

print("\n====================================")
print("Time Features Created")
print("====================================")

print("Total rows:", f"{total_rows:,}")
print("Output file:", output_path)
print(
    "File size (MB):",
    round(os.path.getsize(output_path) / (1024**2), 2)
)

Row group 1/61 processed
Row group 2/61 processed
Row group 3/61 processed
Row group 4/61 processed
Row group 5/61 processed
Row group 6/61 processed
Row group 7/61 processed
Row group 8/61 processed
Row group 9/61 processed
Row group 10/61 processed
Row group 11/61 processed
Row group 12/61 processed
Row group 13/61 processed
Row group 14/61 processed
Row group 15/61 processed
Row group 16/61 processed
Row group 17/61 processed
Row group 18/61 processed
Row group 19/61 processed
Row group 20/61 processed
Row group 21/61 processed
Row group 22/61 processed
Row group 23/61 processed
Row group 24/61 processed
Row group 25/61 processed
Row group 26/61 processed
Row group 27/61 processed
Row group 28/61 processed
Row group 29/61 processed
Row group 30/61 processed
Row group 31/61 processed
Row group 32/61 processed
Row group 33/61 processed
Row group 34/61 processed
Row group 35/61 processed
Row group 36/61 processed
Row group 37/61 processed
Row group 38/61 processed
Row group 39/61 proce

In [5]:
# Open the newly created feature dataset
time_file = pq.ParquetFile("features_time.parquet")

print("====================================")
print("Time Feature Verification")
print("====================================")

print("\nRows:")
print(f"{time_file.metadata.num_rows:,}")

print("\nColumns:")
print(time_file.schema.names)

# Read first row group only
sample_time = time_file.read_row_group(0).to_pandas()

print("\n------------------------------------")
print("New Time Features")
print("------------------------------------")

print(
    sample_time[
        [
            "date",
            "weekday",
            "wday",
            "month",
            "year",
            "day_of_month",
            "week_of_year",
            "day_of_year",
            "quarter",
            "is_weekend"
        ]
    ].head(10)
)

print("\n------------------------------------")
print("Unique Values")
print("------------------------------------")

print(
    "day_of_month:",
    sorted(sample_time["day_of_month"].unique())
)

print(
    "week_of_year:",
    sorted(sample_time["week_of_year"].unique())
)

print(
    "quarter:",
    sorted(sample_time["quarter"].unique())
)

print(
    "is_weekend:",
    sorted(sample_time["is_weekend"].unique())
)

print("\n------------------------------------")
print("Missing Values")
print("------------------------------------")

new_features = [
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend"
]

print(
    sample_time[new_features].isna().sum()
)

print("\n------------------------------------")
print("Final Checks")
print("------------------------------------")

print(
    "Row count preserved:",
    time_file.metadata.num_rows == 58_327_370
)

print(
    "All new features present:",
    all(
        feature in time_file.schema.names
        for feature in new_features
    )
)

print(
    "No missing values in new features:",
    sample_time[new_features].isna().sum().sum() == 0
)

Time Feature Verification

Rows:
58,327,370

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available', 'day_of_month', 'week_of_year', 'day_of_year', 'quarter', 'is_weekend']

------------------------------------
New Time Features
------------------------------------
        date   weekday  wday  month  year  day_of_month  week_of_year  \
0 2011-01-29  Saturday     1      1  2011            29             4   
1 2011-01-29  Saturday     1      1  2011            29             4   
2 2011-01-29  Saturday     1      1  2011            29             4   
3 2011-01-29  Saturday     1      1  2011            29             4   
4 2011-01-29  Saturday     1      1  2011            29             4   
5 2011-01-29  Saturday     1      1  2011            29             4   
6 2011-

In [6]:
# Read a small sample for lag-feature strategy testing

lag_test = time_file.read_row_group(
    0,
    columns=[
        "item_id",
        "store_id",
        "date",
        "sales"
    ]
).to_pandas()

print("====================================")
print("Lag Feature Strategy Test")
print("====================================")

print("\nShape:")
print(lag_test.shape)

print("\nUnique item-store combinations:")
print(
    lag_test[
        ["item_id", "store_id"]
    ].drop_duplicates().shape[0]
)

print("\nDate range:")
print(
    lag_test["date"].min(),
    "to",
    lag_test["date"].max()
)

print("\nFirst rows:")
print(
    lag_test[
        ["item_id", "store_id", "date", "sales"]
    ].head(10)
)

Lag Feature Strategy Test

Shape:
(1048576, 4)

Unique item-store combinations:
1000

Date range:
2011-01-29 00:00:00 to 2013-12-12 00:00:00

First rows:
         item_id store_id       date  sales
0  HOBBIES_1_001     CA_1 2011-01-29      0
1  HOBBIES_1_002     CA_1 2011-01-29      0
2  HOBBIES_1_003     CA_1 2011-01-29      0
3  HOBBIES_1_004     CA_1 2011-01-29      0
4  HOBBIES_1_005     CA_1 2011-01-29      0
5  HOBBIES_1_006     CA_1 2011-01-29      0
6  HOBBIES_1_007     CA_1 2011-01-29      0
7  HOBBIES_1_008     CA_1 2011-01-29     12
8  HOBBIES_1_009     CA_1 2011-01-29      2
9  HOBBIES_1_010     CA_1 2011-01-29      0


In [7]:
# Read a manageable sample from the first row group

lag_check = time_file.read_row_group(
    0,
    columns=[
        "item_id",
        "store_id",
        "date",
        "sales"
    ]
).to_pandas()

# Sort by series and date
lag_check = lag_check.sort_values(
    ["item_id", "store_id", "date"]
).reset_index(drop=True)

# Calculate date difference within each item-store series
lag_check["date_diff"] = (
    lag_check
    .groupby(["item_id", "store_id"])["date"]
    .diff()
    .dt.days
)

print("====================================")
print("Time Continuity Check")
print("====================================")

print("\nDate difference distribution:")
print(
    lag_check["date_diff"]
    .value_counts(dropna=False)
    .sort_index()
)

print("\nNumber of gaps greater than 1 day:")
print(
    (lag_check["date_diff"] > 1).sum()
)

print("\nNumber of negative gaps:")
print(
    (lag_check["date_diff"] < 0).sum()
)

print("\nExample rows:")
print(
    lag_check[
        [
            "item_id",
            "store_id",
            "date",
            "sales",
            "date_diff"
        ]
    ].head(15)
)

Time Continuity Check

Date difference distribution:
date_diff
1.0    1047576
NaN       1000
Name: count, dtype: int64

Number of gaps greater than 1 day:
0

Number of negative gaps:
0

Example rows:
          item_id store_id       date  sales  date_diff
0   HOBBIES_1_001     CA_1 2011-01-29      0        NaN
1   HOBBIES_1_001     CA_1 2011-01-30      0        1.0
2   HOBBIES_1_001     CA_1 2011-01-31      0        1.0
3   HOBBIES_1_001     CA_1 2011-02-01      0        1.0
4   HOBBIES_1_001     CA_1 2011-02-02      0        1.0
5   HOBBIES_1_001     CA_1 2011-02-03      0        1.0
6   HOBBIES_1_001     CA_1 2011-02-04      0        1.0
7   HOBBIES_1_001     CA_1 2011-02-05      0        1.0
8   HOBBIES_1_001     CA_1 2011-02-06      0        1.0
9   HOBBIES_1_001     CA_1 2011-02-07      0        1.0
10  HOBBIES_1_001     CA_1 2011-02-08      0        1.0
11  HOBBIES_1_001     CA_1 2011-02-09      0        1.0
12  HOBBIES_1_001     CA_1 2011-02-10      0        1.0
13  HOBBIES_1_00

In [8]:
# Select one complete item-store series from the sample
test_series = lag_check[
    (lag_check["item_id"] == "HOBBIES_1_001") &
    (lag_check["store_id"] == "CA_1")
].copy()

# Sort chronologically
test_series = test_series.sort_values("date").reset_index(drop=True)

# Create lag features
test_series["lag_1"] = test_series["sales"].shift(1)
test_series["lag_7"] = test_series["sales"].shift(7)

print("====================================")
print("Lag Feature Test")
print("====================================")

print("\nSeries:")
print("item_id:", test_series["item_id"].iloc[0])
print("store_id:", test_series["store_id"].iloc[0])

print("\nRows:", len(test_series))

print("\nFirst 15 observations:")
print(
    test_series[
        [
            "date",
            "sales",
            "lag_1",
            "lag_7"
        ]
    ].head(15)
)

print("\nMissing lag_1:")
print(test_series["lag_1"].isna().sum())

print("\nMissing lag_7:")
print(test_series["lag_7"].isna().sum())

Lag Feature Test

Series:
item_id: HOBBIES_1_001
store_id: CA_1

Rows: 1049

First 15 observations:
         date  sales  lag_1  lag_7
0  2011-01-29      0    NaN    NaN
1  2011-01-30      0    0.0    NaN
2  2011-01-31      0    0.0    NaN
3  2011-02-01      0    0.0    NaN
4  2011-02-02      0    0.0    NaN
5  2011-02-03      0    0.0    NaN
6  2011-02-04      0    0.0    NaN
7  2011-02-05      0    0.0    0.0
8  2011-02-06      0    0.0    0.0
9  2011-02-07      0    0.0    0.0
10 2011-02-08      0    0.0    0.0
11 2011-02-09      0    0.0    0.0
12 2011-02-10      0    0.0    0.0
13 2011-02-11      0    0.0    0.0
14 2011-02-12      0    0.0    0.0

Missing lag_1:
1

Missing lag_7:
7


In [9]:
# Read the first two row groups
group_1 = time_file.read_row_group(
    0,
    columns=["item_id", "store_id", "date", "sales"]
).to_pandas()

group_2 = time_file.read_row_group(
    1,
    columns=["item_id", "store_id", "date", "sales"]
).to_pandas()

print("====================================")
print("Row-Group Boundary Test")
print("====================================")

print("\nGroup 1 last 5 rows:")
print(
    group_1[
        ["item_id", "store_id", "date", "sales"]
    ].tail()
)

print("\nGroup 2 first 5 rows:")
print(
    group_2[
        ["item_id", "store_id", "date", "sales"]
    ].head()
)

# Check whether the first series in group 2
# also exists at the end of group 1
first_item = group_2["item_id"].iloc[0]
first_store = group_2["store_id"].iloc[0]

previous_rows = group_1[
    (group_1["item_id"] == first_item) &
    (group_1["store_id"] == first_store)
]

print("\n------------------------------------")
print("Boundary Series Check")
print("------------------------------------")

print("Item:", first_item)
print("Store:", first_store)

print("\nPrevious observations available in Group 1:")
print(
    previous_rows[
        ["date", "sales"]
    ].tail(10)
)

print("\nNumber of previous observations:")
print(len(previous_rows))

Row-Group Boundary Test

Group 1 last 5 rows:
                 item_id store_id       date  sales
1048571  HOUSEHOLD_1_007     CA_1 2013-12-12      1
1048572  HOUSEHOLD_1_008     CA_1 2013-12-12      0
1048573  HOUSEHOLD_1_009     CA_1 2013-12-12      0
1048574  HOUSEHOLD_1_010     CA_1 2013-12-12      0
1048575  HOUSEHOLD_1_011     CA_1 2013-12-12      1

Group 2 first 5 rows:
           item_id store_id       date  sales
0  HOUSEHOLD_1_012     CA_1 2013-12-12      0
1  HOUSEHOLD_1_013     CA_1 2013-12-12      0
2  HOUSEHOLD_1_014     CA_1 2013-12-12      1
3  HOUSEHOLD_1_015     CA_1 2013-12-12      0
4  HOUSEHOLD_1_016     CA_1 2013-12-12      0

------------------------------------
Boundary Series Check
------------------------------------
Item: HOUSEHOLD_1_012
Store: CA_1

Previous observations available in Group 1:
              date  sales
1038576 2013-12-02      0
1039576 2013-12-03      0
1040576 2013-12-04      0
1041576 2013-12-05      0
1042576 2013-12-06      1
1043576 201

In [10]:
import pyarrow as pa
import pyarrow.parquet as pq
import gc
import os

input_path = "features_time.parquet"
output_path = "features_lag.parquet"

input_file = pq.ParquetFile(input_path)

writer = None
total_rows = 0

# Number of previous observations required
MAX_LAG = 28

# Store the last 28 observations for each item-store series
history = {}

for group_num in range(input_file.num_row_groups):

    df = input_file.read_row_group(group_num).to_pandas()

    # IMPORTANT:
    # Row groups are not guaranteed to end/start at series boundaries.
    # Therefore sort every group by series and date.
    df = df.sort_values(
        ["item_id", "store_id", "date"]
    ).reset_index(drop=True)

    # Create empty lag columns
    df["lag_1"] = np.nan
    df["lag_7"] = np.nan
    df["lag_14"] = np.nan
    df["lag_28"] = np.nan

    # Process each item-store series inside the current group
    for (item_id, store_id), group_indices in df.groupby(
        ["item_id", "store_id"],
        sort=False
    ).groups.items():

        indices = group_indices.to_numpy()

        current_sales = df.loc[indices, "sales"].to_numpy()

        # Previous history from earlier row groups
        previous_sales = history.get(
            (item_id, store_id),
            []
        )

        combined_sales = np.concatenate(
            [
                np.asarray(previous_sales),
                current_sales
            ]
        )

        history_length = len(previous_sales)

        # Calculate required lags
        for lag in [1, 7, 14, 28]:

            lag_values = np.full(
                len(current_sales),
                np.nan,
                dtype=np.float64
            )

            for i in range(len(current_sales)):

                source_position = (
                    history_length + i - lag
                )

                if source_position >= 0:
                    lag_values[i] = combined_sales[
                        source_position
                    ]

            df.loc[
                indices,
                f"lag_{lag}"
            ] = lag_values

        # Keep only last 28 sales for next row group
        history[
            (item_id, store_id)
        ] = combined_sales[-MAX_LAG:].tolist()

    total_rows += len(df)

    table = pa.Table.from_pandas(
        df,
        preserve_index=False
    )

    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)

    print(
        f"Row group {group_num + 1}/{input_file.num_row_groups} processed"
    )

    del df, table
    gc.collect()

writer.close()

print("\n====================================")
print("Lag Features Created")
print("====================================")

print("Total rows:", f"{total_rows:,}")
print("Output file:", output_path)
print(
    "File size (MB):",
    round(os.path.getsize(output_path) / (1024**2), 2)
)

Row group 1/61 processed
Row group 2/61 processed
Row group 3/61 processed
Row group 4/61 processed
Row group 5/61 processed
Row group 6/61 processed
Row group 7/61 processed
Row group 8/61 processed
Row group 9/61 processed
Row group 10/61 processed
Row group 11/61 processed
Row group 12/61 processed
Row group 13/61 processed
Row group 14/61 processed
Row group 15/61 processed
Row group 16/61 processed
Row group 17/61 processed
Row group 18/61 processed
Row group 19/61 processed
Row group 20/61 processed
Row group 21/61 processed
Row group 22/61 processed
Row group 23/61 processed
Row group 24/61 processed
Row group 25/61 processed
Row group 26/61 processed
Row group 27/61 processed
Row group 28/61 processed
Row group 29/61 processed
Row group 30/61 processed
Row group 31/61 processed
Row group 32/61 processed
Row group 33/61 processed
Row group 34/61 processed
Row group 35/61 processed
Row group 36/61 processed
Row group 37/61 processed
Row group 38/61 processed
Row group 39/61 proce

In [11]:
lag_file = pq.ParquetFile("features_lag.parquet")

print("====================================")
print("Lag Feature Verification")
print("====================================")

print("\nRows:")
print(f"{lag_file.metadata.num_rows:,}")

print("\nColumns:")
print(lag_file.schema.names)

# Read first row group
lag_sample = lag_file.read_row_group(
    0,
    columns=[
        "item_id",
        "store_id",
        "date",
        "sales",
        "lag_1",
        "lag_7",
        "lag_14",
        "lag_28"
    ]
).to_pandas()

print("\n------------------------------------")
print("First 35 observations")
print("------------------------------------")

print(
    lag_sample[
        [
            "item_id",
            "store_id",
            "date",
            "sales",
            "lag_1",
            "lag_7",
            "lag_14",
            "lag_28"
        ]
    ].head(35)
)

print("\n------------------------------------")
print("Missing Values")
print("------------------------------------")

lag_columns = [
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28"
]

print(
    lag_sample[lag_columns].isna().sum()
)

print("\n------------------------------------")
print("Lag Statistics")
print("------------------------------------")

print(
    lag_sample[lag_columns].describe()
)

print("\n------------------------------------")
print("Final Checks")
print("------------------------------------")

print(
    "Row count preserved:",
    lag_file.metadata.num_rows == 58_327_370
)

print(
    "All lag features present:",
    all(
        col in lag_file.schema.names
        for col in lag_columns
    )
)

Lag Feature Verification

Rows:
58,327,370

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available', 'day_of_month', 'week_of_year', 'day_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28']

------------------------------------
First 35 observations
------------------------------------
          item_id store_id       date  sales  lag_1  lag_7  lag_14  lag_28
0   HOBBIES_1_001     CA_1 2011-01-29      0    NaN    NaN     NaN     NaN
1   HOBBIES_1_001     CA_1 2011-01-30      0    0.0    NaN     NaN     NaN
2   HOBBIES_1_001     CA_1 2011-01-31      0    0.0    NaN     NaN     NaN
3   HOBBIES_1_001     CA_1 2011-02-01      0    0.0    NaN     NaN     NaN
4   HOBBIES_1_001     CA_1 2011-02-02      0    0.0    NaN     NaN     NaN
5   HOBBIES_1_001     CA_1 

In [12]:
rolling_test = lag_sample[
    (lag_sample["item_id"] == "HOBBIES_1_001") &
    (lag_sample["store_id"] == "CA_1")
].copy()

rolling_test = rolling_test.sort_values(
    "date"
).reset_index(drop=True)

# Leakage-safe rolling features
rolling_test["rolling_mean_7"] = (
    rolling_test["sales"]
    .shift(1)
    .rolling(7)
    .mean()
)

rolling_test["rolling_std_7"] = (
    rolling_test["sales"]
    .shift(1)
    .rolling(7)
    .std()
)

print("====================================")
print("Rolling Feature Test")
print("====================================")

print("\nFirst 15 observations:")

print(
    rolling_test[
        [
            "date",
            "sales",
            "rolling_mean_7",
            "rolling_std_7"
        ]
    ].head(15)
)

print("\nMissing rolling mean:")
print(
    rolling_test["rolling_mean_7"].isna().sum()
)

print("\nMissing rolling std:")
print(
    rolling_test["rolling_std_7"].isna().sum()
)

Rolling Feature Test

First 15 observations:
         date  sales  rolling_mean_7  rolling_std_7
0  2011-01-29      0             NaN            NaN
1  2011-01-30      0             NaN            NaN
2  2011-01-31      0             NaN            NaN
3  2011-02-01      0             NaN            NaN
4  2011-02-02      0             NaN            NaN
5  2011-02-03      0             NaN            NaN
6  2011-02-04      0             NaN            NaN
7  2011-02-05      0             0.0            0.0
8  2011-02-06      0             0.0            0.0
9  2011-02-07      0             0.0            0.0
10 2011-02-08      0             0.0            0.0
11 2011-02-09      0             0.0            0.0
12 2011-02-10      0             0.0            0.0
13 2011-02-11      0             0.0            0.0
14 2011-02-12      0             0.0            0.0

Missing rolling mean:
7

Missing rolling std:
7


In [14]:
import pyarrow as pa
import pyarrow.parquet as pq
import gc
import os
import numpy as np
import pandas as pd

input_path = "features_lag.parquet"
output_path = "features_rolling.parquet"

input_file = pq.ParquetFile(input_path)

writer = None
total_rows = 0

MAX_ROLLING = 28
total_groups = input_file.num_row_groups

# Store previous 28 sales observations for every item-store series
history = {}

for group_num in range(total_groups):

    df = input_file.read_row_group(group_num).to_pandas()

    # Sort chronologically within each item-store series
    df = df.sort_values(
        ["item_id", "store_id", "date"]
    ).reset_index(drop=True)

    # Initialize rolling features
    df["rolling_mean_7"] = np.nan
    df["rolling_mean_28"] = np.nan
    df["rolling_std_7"] = np.nan
    df["rolling_std_28"] = np.nan

    # Process each item-store series
    for (item_id, store_id), group_indices in df.groupby(
        ["item_id", "store_id"],
        sort=False
    ).groups.items():

        indices = group_indices.to_numpy()

        current_sales = df.loc[
            indices, "sales"
        ].to_numpy(dtype=np.float64)

        previous_sales = np.asarray(
            history.get((item_id, store_id), []),
            dtype=np.float64
        )

        # Combine previous history + current group
        combined_sales = np.concatenate(
            [previous_sales, current_sales]
        )

        history_length = len(previous_sales)

        # --------------------------------
        # Vectorized rolling calculations
        # --------------------------------

        series = pd.Series(combined_sales)

        # shift(1) equivalent:
        # rolling window ending BEFORE current observation
        mean_7 = (
            series
            .shift(1)
            .rolling(window=7, min_periods=7)
            .mean()
            .to_numpy()
        )

        mean_28 = (
            series
            .shift(1)
            .rolling(window=28, min_periods=28)
            .mean()
            .to_numpy()
        )

        std_7 = (
            series
            .shift(1)
            .rolling(window=7, min_periods=7)
            .std(ddof=1)
            .to_numpy()
        )

        std_28 = (
            series
            .shift(1)
            .rolling(window=28, min_periods=28)
            .std(ddof=1)
            .to_numpy()
        )

        # Only keep values corresponding to current rows
        start = history_length
        end = history_length + len(current_sales)

        df.loc[
            indices, "rolling_mean_7"
        ] = mean_7[start:end]

        df.loc[
            indices, "rolling_mean_28"
        ] = mean_28[start:end]

        df.loc[
            indices, "rolling_std_7"
        ] = std_7[start:end]

        df.loc[
            indices, "rolling_std_28"
        ] = std_28[start:end]

        # Keep latest 28 observations for next row group
        history[
            (item_id, store_id)
        ] = combined_sales[-MAX_ROLLING:].tolist()

    # --------------------------------
    # Write current row group
    # --------------------------------

    total_rows += len(df)

    table = pa.Table.from_pandas(
        df,
        preserve_index=False
    )

    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)

    # --------------------------------
    # Progress
    # --------------------------------

    progress = (
        (group_num + 1) / total_groups
    ) * 100

    print(
        f"Row group {group_num + 1}/{total_groups} "
        f"| Progress: {progress:.2f}%"
    )

    del df, table
    gc.collect()

writer.close()

print("\n====================================")
print("Rolling Features Created")
print("====================================")

print(
    "Total rows:",
    f"{total_rows:,}"
)

print(
    "Output file:",
    output_path
)

print(
    "File size (MB):",
    round(
        os.path.getsize(output_path) / (1024**2),
        2
    )
)

print("Progress: 100.00%")

Row group 1/61 | Progress: 1.64%
Row group 2/61 | Progress: 3.28%
Row group 3/61 | Progress: 4.92%
Row group 4/61 | Progress: 6.56%
Row group 5/61 | Progress: 8.20%
Row group 6/61 | Progress: 9.84%
Row group 7/61 | Progress: 11.48%
Row group 8/61 | Progress: 13.11%
Row group 9/61 | Progress: 14.75%
Row group 10/61 | Progress: 16.39%
Row group 11/61 | Progress: 18.03%
Row group 12/61 | Progress: 19.67%
Row group 13/61 | Progress: 21.31%
Row group 14/61 | Progress: 22.95%
Row group 15/61 | Progress: 24.59%
Row group 16/61 | Progress: 26.23%
Row group 17/61 | Progress: 27.87%
Row group 18/61 | Progress: 29.51%
Row group 19/61 | Progress: 31.15%
Row group 20/61 | Progress: 32.79%
Row group 21/61 | Progress: 34.43%
Row group 22/61 | Progress: 36.07%
Row group 23/61 | Progress: 37.70%
Row group 24/61 | Progress: 39.34%
Row group 25/61 | Progress: 40.98%
Row group 26/61 | Progress: 42.62%
Row group 27/61 | Progress: 44.26%
Row group 28/61 | Progress: 45.90%
Row group 29/61 | Progress: 47.54%


In [15]:
rolling_file = pq.ParquetFile(
    "features_rolling.parquet"
)

print("====================================")
print("Rolling Features Verification")
print("====================================")

print("\nRows:")
print(f"{rolling_file.metadata.num_rows:,}")

print("\nColumns:")
print(rolling_file.schema.names)

print("\nRow groups:")
print(rolling_file.num_row_groups)

Rolling Features Verification

Rows:
58,327,370

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available', 'day_of_month', 'week_of_year', 'day_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28']

Row groups:
61


In [16]:
# Read first row group
rolling_sample = rolling_file.read_row_group(
    0,
    columns=[
        "item_id",
        "store_id",
        "date",
        "sales",
        "rolling_mean_7",
        "rolling_mean_28",
        "rolling_std_7",
        "rolling_std_28"
    ]
).to_pandas()

# Select one complete series
rolling_sample = rolling_sample[
    (rolling_sample["item_id"] == "HOBBIES_1_001") &
    (rolling_sample["store_id"] == "CA_1")
].copy()

rolling_sample = rolling_sample.sort_values(
    "date"
).reset_index(drop=True)

print("====================================")
print("Rolling Values Verification")
print("====================================")

print("\nFirst 35 observations:")

print(
    rolling_sample[
        [
            "date",
            "sales",
            "rolling_mean_7",
            "rolling_mean_28",
            "rolling_std_7",
            "rolling_std_28"
        ]
    ].head(35)
)

print("\nMissing values:")

print(
    rolling_sample[
        [
            "rolling_mean_7",
            "rolling_mean_28",
            "rolling_std_7",
            "rolling_std_28"
        ]
    ].isna().sum()
)

Rolling Values Verification

First 35 observations:
         date  sales  rolling_mean_7  rolling_mean_28  rolling_std_7  \
0  2011-01-29      0             NaN              NaN            NaN   
1  2011-01-30      0             NaN              NaN            NaN   
2  2011-01-31      0             NaN              NaN            NaN   
3  2011-02-01      0             NaN              NaN            NaN   
4  2011-02-02      0             NaN              NaN            NaN   
5  2011-02-03      0             NaN              NaN            NaN   
6  2011-02-04      0             NaN              NaN            NaN   
7  2011-02-05      0             0.0              NaN            0.0   
8  2011-02-06      0             0.0              NaN            0.0   
9  2011-02-07      0             0.0              NaN            0.0   
10 2011-02-08      0             0.0              NaN            0.0   
11 2011-02-09      0             0.0              NaN            0.0   
12 2011-02-1

In [17]:
# Read the end of row group 1 and beginning of row group 2

group_1 = rolling_file.read_row_group(
    0,
    columns=[
        "item_id",
        "store_id",
        "date",
        "sales",
        "rolling_mean_7",
        "rolling_mean_28"
    ]
).to_pandas()

group_2 = rolling_file.read_row_group(
    1,
    columns=[
        "item_id",
        "store_id",
        "date",
        "sales",
        "rolling_mean_7",
        "rolling_mean_28"
    ]
).to_pandas()

# Identify the first series appearing in row group 2
first_item = group_2["item_id"].iloc[0]
first_store = group_2["store_id"].iloc[0]

previous_rows = group_1[
    (group_1["item_id"] == first_item) &
    (group_1["store_id"] == first_store)
].copy()

current_rows = group_2[
    (group_2["item_id"] == first_item) &
    (group_2["store_id"] == first_store)
].copy()

previous_rows = previous_rows.sort_values("date")
current_rows = current_rows.sort_values("date")

print("====================================")
print("Row Group Boundary Check")
print("====================================")

print("\nSeries:")
print(first_item, "|", first_store)

print("\nLast 10 rows from previous row group:")

print(
    previous_rows[
        [
            "date",
            "sales",
            "rolling_mean_7",
            "rolling_mean_28"
        ]
    ].tail(10)
)

print("\nFirst 10 rows from current row group:")

print(
    current_rows[
        [
            "date",
            "sales",
            "rolling_mean_7",
            "rolling_mean_28"
        ]
    ].head(10)
)

print("\nPrevious rows available:")
print(len(previous_rows))

print("\nCurrent rows available:")
print(len(current_rows))

Row Group Boundary Check

Series:
HOBBIES_1_001 | CA_1

Last 10 rows from previous row group:
           date  sales  rolling_mean_7  rolling_mean_28
1039 2013-12-03      0        0.857143         0.750000
1040 2013-12-04      1        0.857143         0.714286
1041 2013-12-05      0        1.000000         0.714286
1042 2013-12-06      1        0.857143         0.678571
1043 2013-12-07      1        0.857143         0.678571
1044 2013-12-08      0        0.714286         0.714286
1045 2013-12-09      1        0.571429         0.642857
1046 2013-12-10      1        0.571429         0.571429
1047 2013-12-11      0        0.714286         0.571429
1048 2013-12-12      3        0.571429         0.535714

First 10 rows from current row group:
        date  sales  rolling_mean_7  rolling_mean_28
0 2013-12-13      0        1.000000         0.642857
1 2013-12-14      0        0.857143         0.642857
2 2013-12-15      3        0.714286         0.642857
3 2013-12-16      0        1.142857    

In [18]:
# Read one row group
price_test = rolling_file.read_row_group(
    0,
    columns=[
        "item_id",
        "store_id",
        "date",
        "sell_price",
        "price_available"
    ]
).to_pandas()

# Select one item-store series
price_test = price_test[
    (price_test["item_id"] == "HOBBIES_1_001") &
    (price_test["store_id"] == "CA_1")
].copy()

price_test = price_test.sort_values(
    "date"
).reset_index(drop=True)

# Make sure price is numeric
price_test["sell_price"] = pd.to_numeric(
    price_test["sell_price"],
    errors="coerce"
)

# --------------------------------
# Price Change
# --------------------------------

price_test["price_change_1"] = (
    price_test["sell_price"]
    - price_test["sell_price"].shift(1)
)

# --------------------------------
# Price Change Percentage
# --------------------------------

previous_price = price_test["sell_price"].shift(1)

price_test["price_change_pct_1"] = np.where(
    (previous_price.notna()) &
    (previous_price != 0) &
    (price_test["sell_price"].notna()),
    (
        (price_test["sell_price"] - previous_price)
        / previous_price
    ),
    np.nan
)

# --------------------------------
# Price Relative to Previous
# 7-Day Average
# --------------------------------

price_test["previous_7day_avg_price"] = (
    price_test["sell_price"]
    .shift(1)
    .rolling(7, min_periods=7)
    .mean()
)

price_test["price_relative_7"] = (
    price_test["sell_price"]
    / price_test["previous_7day_avg_price"]
)

# --------------------------------
# Print Results
# --------------------------------

print("====================================")
print("Price Feature Test")
print("====================================")

print("\nFirst 20 observations:")

print(
    price_test[
        [
            "date",
            "sell_price",
            "price_available",
            "price_change_1",
            "price_change_pct_1",
            "previous_7day_avg_price",
            "price_relative_7"
        ]
    ].head(20)
)

print("\nMissing values:")

print(
    price_test[
        [
            "price_change_1",
            "price_change_pct_1",
            "previous_7day_avg_price",
            "price_relative_7"
        ]
    ].isna().sum()
)

Price Feature Test

First 20 observations:
         date  sell_price  price_available  price_change_1  \
0  2011-01-29         NaN                0             NaN   
1  2011-01-30         NaN                0             NaN   
2  2011-01-31         NaN                0             NaN   
3  2011-02-01         NaN                0             NaN   
4  2011-02-02         NaN                0             NaN   
5  2011-02-03         NaN                0             NaN   
6  2011-02-04         NaN                0             NaN   
7  2011-02-05         NaN                0             NaN   
8  2011-02-06         NaN                0             NaN   
9  2011-02-07         NaN                0             NaN   
10 2011-02-08         NaN                0             NaN   
11 2011-02-09         NaN                0             NaN   
12 2011-02-10         NaN                0             NaN   
13 2011-02-11         NaN                0             NaN   
14 2011-02-12         NaN  

In [19]:
# Find a series with sufficient price availability
price_test_all = rolling_file.read_row_group(
    0,
    columns=[
        "item_id",
        "store_id",
        "date",
        "sell_price",
        "price_available"
    ]
).to_pandas()

# Find item-store series having at least 15 price observations
series_price_counts = (
    price_test_all
    .groupby(["item_id", "store_id"])["price_available"]
    .sum()
    .sort_values(ascending=False)
)

test_item, test_store = series_price_counts.index[0]

print("Selected test series:")
print("Item :", test_item)
print("Store:", test_store)

price_test = price_test_all[
    (price_test_all["item_id"] == test_item) &
    (price_test_all["store_id"] == test_store)
].copy()

price_test = price_test.sort_values(
    "date"
).reset_index(drop=True)

price_test["sell_price"] = pd.to_numeric(
    price_test["sell_price"],
    errors="coerce"
)

# --------------------------------
# Previous AVAILABLE price
# --------------------------------

previous_available_price = (
    price_test["sell_price"]
    .ffill()
    .shift(1)
)

# --------------------------------
# Price Change
# --------------------------------

price_test["price_change_1"] = np.where(
    price_test["price_available"] == 1,
    price_test["sell_price"] - previous_available_price,
    np.nan
)

# --------------------------------
# Price Change Percentage
# --------------------------------

price_test["price_change_pct_1"] = np.where(
    (price_test["price_available"] == 1) &
    (previous_available_price.notna()) &
    (previous_available_price != 0),
    (
        (price_test["sell_price"] - previous_available_price)
        / previous_available_price
    ),
    np.nan
)

# --------------------------------
# Previous 7 available prices
# --------------------------------

previous_7_prices = (
    price_test["sell_price"]
    .where(price_test["price_available"] == 1)
    .ffill()
    .shift(1)
    .rolling(7, min_periods=7)
    .mean()
)

price_test["previous_7day_avg_price"] = previous_7_prices

price_test["price_relative_7"] = np.where(
    (price_test["price_available"] == 1) &
    (price_test["previous_7day_avg_price"].notna()) &
    (price_test["previous_7day_avg_price"] != 0),
    (
        price_test["sell_price"]
        / price_test["previous_7day_avg_price"]
    ),
    np.nan
)

# --------------------------------
# Display
# --------------------------------

print("\n====================================")
print("Price Feature Test")
print("====================================")

print(
    price_test[
        [
            "date",
            "sell_price",
            "price_available",
            "price_change_1",
            "price_change_pct_1",
            "previous_7day_avg_price",
            "price_relative_7"
        ]
    ].head(30)
)

Selected test series:
Item : HOBBIES_1_010
Store: CA_1

Price Feature Test
         date  sell_price  price_available  price_change_1  \
0  2011-01-29        3.17                1             NaN   
1  2011-01-30        3.17                1             0.0   
2  2011-01-31        3.17                1             0.0   
3  2011-02-01        3.17                1             0.0   
4  2011-02-02        3.17                1             0.0   
5  2011-02-03        3.17                1             0.0   
6  2011-02-04        3.17                1             0.0   
7  2011-02-05        3.17                1             0.0   
8  2011-02-06        3.17                1             0.0   
9  2011-02-07        3.17                1             0.0   
10 2011-02-08        3.17                1             0.0   
11 2011-02-09        3.17                1             0.0   
12 2011-02-10        3.17                1             0.0   
13 2011-02-11        3.17                1             0.

In [21]:
# Read one row group
price_gap_test = rolling_file.read_row_group(
    0,
    columns=[
        "item_id",
        "store_id",
        "date",
        "sell_price",
        "price_available"
    ]
).to_pandas()

price_gap_test["sell_price"] = pd.to_numeric(
    price_gap_test["sell_price"],
    errors="coerce"
)

# Find a series containing both available and missing prices
series_stats = (
    price_gap_test
    .groupby(["item_id", "store_id"])
    .agg(
        total_rows=("price_available", "size"),
        available_prices=("price_available", "sum")
    )
)

# Select a series where price is sometimes missing
candidate_series = series_stats[
    (series_stats["available_prices"] > 0) &
    (series_stats["available_prices"] < series_stats["total_rows"])
]

test_item, test_store = candidate_series.index[0]

price_gap_test = price_gap_test[
    (price_gap_test["item_id"] == test_item) &
    (price_gap_test["store_id"] == test_store)
].copy()

price_gap_test = price_gap_test.sort_values(
    "date"
).reset_index(drop=True)

print("====================================")
print("Missing Price Gap Test")
print("====================================")

print("\nSelected series:")
print("Item :", test_item)
print("Store:", test_store)

print("\nPrice availability:")
print(
    price_gap_test["price_available"]
    .value_counts()
)

# --------------------------------
# Previous AVAILABLE price
# --------------------------------

previous_available_price = (
    price_gap_test["sell_price"]
    .ffill()
    .shift(1)
)

price_gap_test["previous_available_price"] = (
    previous_available_price
)

# --------------------------------
# Price change
# --------------------------------

price_gap_test["price_change_1"] = np.where(
    (price_gap_test["price_available"] == 1) &
    (previous_available_price.notna()),
    (
        price_gap_test["sell_price"]
        - previous_available_price
    ),
    np.nan
)

# --------------------------------
# Show transitions
# --------------------------------

transition_mask = (
    (price_gap_test["price_available"] == 1) &
    (
        price_gap_test["price_available"].shift(1) == 0
    )
)

transition_rows = price_gap_test[
    transition_mask
].copy()

print("\n====================================")
print("Missing → Available Price Transitions")
print("====================================")

print(
    transition_rows[
        [
            "date",
            "sell_price",
            "price_available",
            "previous_available_price",
            "price_change_1"
        ]
    ].head(10)
)

Missing Price Gap Test

Selected series:
Item : HOBBIES_1_001
Store: CA_1

Price availability:
price_available
0    896
1    153
Name: count, dtype: int64

Missing → Available Price Transitions
          date  sell_price  price_available  previous_available_price  \
896 2013-07-13        9.58                1                       NaN   

     price_change_1  
896             NaN  


In [22]:
# Scan row groups to find a real:
# Available → Missing → Available transition

transition_found = False

for group_num in range(rolling_file.num_row_groups):

    df_check = rolling_file.read_row_group(
        group_num,
        columns=[
            "item_id",
            "store_id",
            "date",
            "sell_price",
            "price_available"
        ]
    ).to_pandas()

    df_check = df_check.sort_values(
        ["item_id", "store_id", "date"]
    ).reset_index(drop=True)

    # Previous available price within each series
    df_check["previous_available_price"] = (
        df_check
        .groupby(["item_id", "store_id"])["sell_price"]
        .ffill()
        .groupby(
            [df_check["item_id"], df_check["store_id"]]
        )
        .shift(1)
    )

    # Find rows where:
    # current price available
    # previous day price unavailable
    # previous available price exists

    transition_mask = (
        (df_check["price_available"] == 1) &
        (df_check["price_available"].shift(1) == 0) &
        (df_check["previous_available_price"].notna())
    )

    transitions = df_check[transition_mask]

    if len(transitions) > 0:

        print("====================================")
        print("Real Price Gap Found")
        print("====================================")

        print(
            transitions[
                [
                    "item_id",
                    "store_id",
                    "date",
                    "sell_price",
                    "price_available",
                    "previous_available_price"
                ]
            ].head(5)
        )

        transition_found = True
        break

    del df_check
    gc.collect()

if not transition_found:

    print(
        "No Available → Missing → Available "
        "transition found in the scanned row groups."
    )

No Available → Missing → Available transition found in the scanned row groups.


In [23]:
import pyarrow as pa
import pyarrow.parquet as pq
import gc
import os
import numpy as np
import pandas as pd

input_path = "features_rolling.parquet"
output_path = "features_price.parquet"

input_file = pq.ParquetFile(input_path)

writer = None
total_rows = 0
total_groups = input_file.num_row_groups

# Store previous available prices for each item-store series
history = {}

for group_num in range(total_groups):

    df = input_file.read_row_group(
        group_num
    ).to_pandas()

    # Keep chronological order
    df = df.sort_values(
        ["item_id", "store_id", "date"]
    ).reset_index(drop=True)

    # Initialize features
    df["price_change_1"] = np.nan
    df["price_change_pct_1"] = np.nan
    df["price_relative_7"] = np.nan

    # Process each item-store series
    for (item_id, store_id), group_indices in df.groupby(
        ["item_id", "store_id"],
        sort=False
    ).groups.items():

        indices = group_indices.to_numpy()

        current_price = pd.to_numeric(
            df.loc[indices, "sell_price"],
            errors="coerce"
        ).to_numpy(dtype=np.float64)

        current_available = df.loc[
            indices, "price_available"
        ].to_numpy(dtype=np.int8)

        # Previous available price history
        previous_prices = np.asarray(
            history.get(
                (item_id, store_id),
                []
            ),
            dtype=np.float64
        )

        # Current valid prices only
        valid_mask = (
            current_available == 1
        )

        valid_prices = current_price[
            valid_mask
        ]

        # Combine previous + current available prices
        combined_prices = np.concatenate(
            [
                previous_prices,
                valid_prices
            ]
        )

        history_length = len(
            previous_prices
        )

        # ====================================
        # Price features
        # ====================================

        if len(valid_prices) > 0:

            # --------------------------------
            # Previous available price
            # --------------------------------

            previous_price = np.full(
                len(valid_prices),
                np.nan,
                dtype=np.float64
            )

            if len(combined_prices) > 1:

                previous_price_values = (
                    combined_prices[:-1]
                )

                current_positions = np.arange(
                    history_length,
                    history_length + len(valid_prices)
                )

                previous_price = (
                    combined_prices[
                        current_positions - 1
                    ]
                )

            # --------------------------------
            # Price change
            # --------------------------------

            price_change = (
                valid_prices
                - previous_price
            )

            # First-ever available price has no
            # previous price
            price_change[
                np.isnan(previous_price)
            ] = np.nan

            # --------------------------------
            # Percentage change
            # --------------------------------

            price_change_pct = np.full(
                len(valid_prices),
                np.nan,
                dtype=np.float64
            )

            valid_previous = (
                ~np.isnan(previous_price)
                & (previous_price != 0)
            )

            price_change_pct[
                valid_previous
            ] = (
                (
                    valid_prices[valid_previous]
                    - previous_price[valid_previous]
                )
                / previous_price[valid_previous]
            )

            # --------------------------------
            # Previous 7 available prices
            # --------------------------------

            rolling_mean_7 = (
                pd.Series(combined_prices)
                .shift(1)
                .rolling(
                    window=7,
                    min_periods=7
                )
                .mean()
                .to_numpy()
            )

            current_positions = np.arange(
                history_length,
                history_length + len(valid_prices)
            )

            previous_7_avg = (
                rolling_mean_7[
                    current_positions
                ]
            )

            # --------------------------------
            # Price relative to previous 7
            # --------------------------------

            price_relative_7 = np.full(
                len(valid_prices),
                np.nan,
                dtype=np.float64
            )

            valid_avg = (
                ~np.isnan(previous_7_avg)
                & (previous_7_avg != 0)
            )

            price_relative_7[
                valid_avg
            ] = (
                valid_prices[valid_avg]
                / previous_7_avg[valid_avg]
            )

            # --------------------------------
            # Assign only to price-available rows
            # --------------------------------

            valid_indices = indices[
                valid_mask
            ]

            df.loc[
                valid_indices,
                "price_change_1"
            ] = price_change

            df.loc[
                valid_indices,
                "price_change_pct_1"
            ] = price_change_pct

            df.loc[
                valid_indices,
                "price_relative_7"
            ] = price_relative_7

        # ====================================
        # Update history
        # ====================================

        history[
            (item_id, store_id)
        ] = combined_prices[-7:].tolist()

    # ====================================
    # Write row group
    # ====================================

    total_rows += len(df)

    table = pa.Table.from_pandas(
        df,
        preserve_index=False
    )

    if writer is None:

        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)

    # ====================================
    # Progress
    # ====================================

    progress = (
        (group_num + 1)
        / total_groups
    ) * 100

    print(
        f"Row group {group_num + 1}/{total_groups} "
        f"| Progress: {progress:.2f}%"
    )

    del df, table
    gc.collect()

writer.close()

print("\n====================================")
print("Price Features Created")
print("====================================")

print(
    "Total rows:",
    f"{total_rows:,}"
)

print(
    "Output file:",
    output_path
)

print(
    "File size (MB):",
    round(
        os.path.getsize(output_path)
        / (1024**2),
        2
    )
)

print("Progress: 100.00%")

Row group 1/61 | Progress: 1.64%
Row group 2/61 | Progress: 3.28%
Row group 3/61 | Progress: 4.92%
Row group 4/61 | Progress: 6.56%
Row group 5/61 | Progress: 8.20%
Row group 6/61 | Progress: 9.84%
Row group 7/61 | Progress: 11.48%
Row group 8/61 | Progress: 13.11%
Row group 9/61 | Progress: 14.75%
Row group 10/61 | Progress: 16.39%
Row group 11/61 | Progress: 18.03%
Row group 12/61 | Progress: 19.67%
Row group 13/61 | Progress: 21.31%
Row group 14/61 | Progress: 22.95%
Row group 15/61 | Progress: 24.59%
Row group 16/61 | Progress: 26.23%
Row group 17/61 | Progress: 27.87%
Row group 18/61 | Progress: 29.51%
Row group 19/61 | Progress: 31.15%
Row group 20/61 | Progress: 32.79%
Row group 21/61 | Progress: 34.43%
Row group 22/61 | Progress: 36.07%
Row group 23/61 | Progress: 37.70%
Row group 24/61 | Progress: 39.34%
Row group 25/61 | Progress: 40.98%
Row group 26/61 | Progress: 42.62%
Row group 27/61 | Progress: 44.26%
Row group 28/61 | Progress: 45.90%
Row group 29/61 | Progress: 47.54%


In [24]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

file = pq.ParquetFile("features_price.parquet")

print("Rows:", f"{file.metadata.num_rows:,}")
print("Row groups:", file.num_row_groups)

print("\nColumns:")
print(file.schema_arrow.names)

# Read one row group for verification
df = file.read_row_group(0).to_pandas()

# --------------------------------------------------
# 1. Check required features
# --------------------------------------------------

price_features = [
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7"
]

print("\nRequired price features present:")
for col in price_features:
    print(f"{col}: {col in df.columns}")

# --------------------------------------------------
# 2. Missing counts
# --------------------------------------------------

print("\nMissing counts in sample row group:")
print(
    df[
        [
            "sell_price",
            "price_available",
            "price_change_1",
            "price_change_pct_1",
            "price_relative_7"
        ]
    ].isna().sum()
)

# --------------------------------------------------
# 3. Check an always-available price series
# --------------------------------------------------

series = df[
    (df["item_id"] == "HOBBIES_1_010") &
    (df["store_id"] == "CA_1")
].copy()

series = series.sort_values("date")

print("\nHOBBIES_1_010 | CA_1")
print(
    series[
        [
            "date",
            "sell_price",
            "price_available",
            "price_change_1",
            "price_change_pct_1",
            "price_relative_7"
        ]
    ].head(15)
)

# --------------------------------------------------
# 4. Price feature statistics
# --------------------------------------------------

print("\nPrice feature statistics:")
print(
    df[price_features].describe()
)

# --------------------------------------------------
# 5. Check price-unavailable rows
# --------------------------------------------------

unavailable = df[df["price_available"] == 0]

print("\nPrice unavailable rows:", len(unavailable))

if len(unavailable) > 0:
    print(
        "\nPrice features when price is unavailable:"
    )
    print(
        unavailable[price_features]
        .notna()
        .sum()
    )

# --------------------------------------------------
# 6. Basic sanity checks
# --------------------------------------------------

print("\nSanity checks:")

print(
    "Any infinite price_change_pct_1:",
    np.isinf(
        df["price_change_pct_1"]
    ).sum()
)

print(
    "Any infinite price_relative_7:",
    np.isinf(
        df["price_relative_7"]
    ).sum()
)

Rows: 58,327,370
Row groups: 61

Columns:
['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'd', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'price_available', 'day_of_month', 'week_of_year', 'day_of_year', 'quarter', 'is_weekend', 'lag_1', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_mean_28', 'rolling_std_7', 'rolling_std_28', 'price_change_1', 'price_change_pct_1', 'price_relative_7']

Required price features present:
price_change_1: True
price_change_pct_1: True
price_relative_7: True

Missing counts in sample row group:
sell_price            367871
price_available            0
price_change_1        367871
price_change_pct_1    367871
price_relative_7      373905
dtype: int64

HOBBIES_1_010 | CA_1
           date  sell_price  price_available  price_change_1  \
9441 2011-01-29        3.17                1             0.2   
9442 

In [25]:
import pyarrow.parquet as pq
import pandas as pd

file = pq.ParquetFile("features_price.parquet")

df = file.read_row_group(0).to_pandas()

event_snap_cols = [
    "date",
    "state_id",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI"
]

print("Event + SNAP sample:")
print(df[event_snap_cols].head(20))

print("\nEvent 1 non-null rows:")
print(df["event_name_1"].notna().sum())

print("\nEvent 2 non-null rows:")
print(df["event_name_2"].notna().sum())

print("\nSNAP columns unique values:")

for col in ["snap_CA", "snap_TX", "snap_WI"]:
    print(f"{col}: {sorted(df[col].dropna().unique())}")

Event + SNAP sample:
         date state_id   event_name_1 event_type_1 event_name_2 event_type_2  \
0  2011-01-29       CA           None         None         None         None   
1  2011-01-30       CA           None         None         None         None   
2  2011-01-31       CA           None         None         None         None   
3  2011-02-01       CA           None         None         None         None   
4  2011-02-02       CA           None         None         None         None   
5  2011-02-03       CA           None         None         None         None   
6  2011-02-04       CA           None         None         None         None   
7  2011-02-05       CA           None         None         None         None   
8  2011-02-06       CA      SuperBowl     Sporting         None         None   
9  2011-02-07       CA           None         None         None         None   
10 2011-02-08       CA           None         None         None         None   
11 2011-02-09      

In [26]:
# Event + SNAP feature test

test = df[
    [
        "date",
        "state_id",
        "event_name_1",
        "event_name_2",
        "snap_CA",
        "snap_TX",
        "snap_WI"
    ]
].copy()

# --------------------------------------------------
# 1. Event day
# --------------------------------------------------

test["is_event_day"] = (
    test["event_name_1"].notna()
    | test["event_name_2"].notna()
).astype("int8")

# --------------------------------------------------
# 2. Number of active events
# --------------------------------------------------

test["event_count"] = (
    test["event_name_1"].notna().astype("int8")
    +
    test["event_name_2"].notna().astype("int8")
)

# --------------------------------------------------
# 3. State-specific SNAP
# --------------------------------------------------

test["snap_active"] = np.select(
    [
        test["state_id"] == "CA",
        test["state_id"] == "TX",
        test["state_id"] == "WI"
    ],
    [
        test["snap_CA"],
        test["snap_TX"],
        test["snap_WI"]
    ],
    default=0
).astype("int8")

# --------------------------------------------------
# Show result
# --------------------------------------------------

print(test.head(20))

print("\nUnique values:")

print(
    "is_event_day:",
    sorted(test["is_event_day"].unique())
)

print(
    "event_count:",
    sorted(test["event_count"].unique())
)

print(
    "snap_active:",
    sorted(test["snap_active"].unique())
)

         date state_id   event_name_1 event_name_2  snap_CA  snap_TX  snap_WI  \
0  2011-01-29       CA           None         None        0        0        0   
1  2011-01-30       CA           None         None        0        0        0   
2  2011-01-31       CA           None         None        0        0        0   
3  2011-02-01       CA           None         None        1        1        0   
4  2011-02-02       CA           None         None        1        0        1   
5  2011-02-03       CA           None         None        1        1        1   
6  2011-02-04       CA           None         None        1        0        0   
7  2011-02-05       CA           None         None        1        1        1   
8  2011-02-06       CA      SuperBowl         None        1        1        1   
9  2011-02-07       CA           None         None        1        1        0   
10 2011-02-08       CA           None         None        1        0        1   
11 2011-02-09       CA      

In [27]:
import pyarrow as pa
import pyarrow.parquet as pq
import gc
import os
import numpy as np
import pandas as pd

input_path = "features_price.parquet"
output_path = "features_event_snap.parquet"

input_file = pq.ParquetFile(input_path)

writer = None
total_rows = 0
total_groups = input_file.num_row_groups

for group_num in range(total_groups):

    df = input_file.read_row_group(
        group_num
    ).to_pandas()

    # ==========================================
    # Event Features
    # ==========================================

    df["is_event_day"] = (
        df["event_name_1"].notna()
        | df["event_name_2"].notna()
    ).astype("int8")

    df["event_count"] = (
        df["event_name_1"].notna().astype("int8")
        +
        df["event_name_2"].notna().astype("int8")
    )

    # ==========================================
    # State-specific SNAP Feature
    # ==========================================

    df["snap_active"] = np.select(
        [
            df["state_id"] == "CA",
            df["state_id"] == "TX",
            df["state_id"] == "WI"
        ],
        [
            df["snap_CA"],
            df["snap_TX"],
            df["snap_WI"]
        ],
        default=0
    ).astype("int8")

    # ==========================================
    # Write row group
    # ==========================================

    total_rows += len(df)

    table = pa.Table.from_pandas(
        df,
        preserve_index=False
    )

    if writer is None:
        writer = pq.ParquetWriter(
            output_path,
            table.schema,
            compression="snappy"
        )

    writer.write_table(table)

    # ==========================================
    # Progress
    # ==========================================

    progress = (
        (group_num + 1)
        / total_groups
    ) * 100

    print(
        f"Row group {group_num + 1}/{total_groups} "
        f"| Progress: {progress:.2f}%"
    )

    del df, table
    gc.collect()

writer.close()

print("\n====================================")
print("Event + SNAP Features Created")
print("====================================")

print(
    "Total rows:",
    f"{total_rows:,}"
)

print(
    "Output file:",
    output_path
)

print(
    "File size (MB):",
    round(
        os.path.getsize(output_path)
        / (1024**2),
        2
    )
)

print("Progress: 100.00%")

Row group 1/61 | Progress: 1.64%
Row group 2/61 | Progress: 3.28%
Row group 3/61 | Progress: 4.92%
Row group 4/61 | Progress: 6.56%
Row group 5/61 | Progress: 8.20%
Row group 6/61 | Progress: 9.84%
Row group 7/61 | Progress: 11.48%
Row group 8/61 | Progress: 13.11%
Row group 9/61 | Progress: 14.75%
Row group 10/61 | Progress: 16.39%
Row group 11/61 | Progress: 18.03%
Row group 12/61 | Progress: 19.67%
Row group 13/61 | Progress: 21.31%
Row group 14/61 | Progress: 22.95%
Row group 15/61 | Progress: 24.59%
Row group 16/61 | Progress: 26.23%
Row group 17/61 | Progress: 27.87%
Row group 18/61 | Progress: 29.51%
Row group 19/61 | Progress: 31.15%
Row group 20/61 | Progress: 32.79%
Row group 21/61 | Progress: 34.43%
Row group 22/61 | Progress: 36.07%
Row group 23/61 | Progress: 37.70%
Row group 24/61 | Progress: 39.34%
Row group 25/61 | Progress: 40.98%
Row group 26/61 | Progress: 42.62%
Row group 27/61 | Progress: 44.26%
Row group 28/61 | Progress: 45.90%
Row group 29/61 | Progress: 47.54%


In [28]:
import pyarrow.parquet as pq
import pandas as pd
import numpy as np

# ==================================================
# Load Parquet metadata
# ==================================================

file = pq.ParquetFile("features_event_snap.parquet")

print("====================================")
print("FINAL FEATURE DATASET VERIFICATION")
print("====================================")

print("\nRows:")
print(f"{file.metadata.num_rows:,}")

print("\nRow groups:")
print(file.num_row_groups)

print("\nColumns:")
columns = file.schema_arrow.names

for i, col in enumerate(columns, 1):
    print(f"{i:2d}. {col}")

# ==================================================
# Expected engineered features
# ==================================================

expected_features = [
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend",
    "lag_1",
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_mean_28",
    "rolling_std_7",
    "rolling_std_28",
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7",
    "is_event_day",
    "event_count",
    "snap_active"
]

print("\n====================================")
print("ENGINEERED FEATURE CHECK")
print("====================================")

for col in expected_features:
    print(f"{col:25s}: {col in columns}")

# ==================================================
# Read one row group for detailed checks
# ==================================================

df = file.read_row_group(0).to_pandas()

# ==================================================
# Row count / column count
# ==================================================

print("\n====================================")
print("BASIC SANITY CHECKS")
print("====================================")

print(
    "Expected rows:",
    f"{58_327_370:,}"
)

print(
    "Actual rows:",
    f"{file.metadata.num_rows:,}"
)

print(
    "Row count preserved:",
    file.metadata.num_rows == 58_327_370
)

print(
    "Total columns:",
    len(columns)
)

# ==================================================
# Missing values
# ==================================================

print("\n====================================")
print("MISSING VALUE CHECK")
print("====================================")

check_columns = [
    "sales",
    "day_of_month",
    "week_of_year",
    "day_of_year",
    "quarter",
    "is_weekend",
    "is_event_day",
    "event_count",
    "snap_active"
]

print(
    df[check_columns].isna().sum()
)

# ==================================================
# Value range checks
# ==================================================

print("\n====================================")
print("VALUE RANGE CHECK")
print("====================================")

print(
    "is_weekend:",
    sorted(df["is_weekend"].unique())
)

print(
    "is_event_day:",
    sorted(df["is_event_day"].unique())
)

print(
    "event_count:",
    sorted(df["event_count"].unique())
)

print(
    "snap_active:",
    sorted(df["snap_active"].unique())
)

print(
    "lag_1 minimum:",
    df["lag_1"].min()
)

print(
    "lag_7 minimum:",
    df["lag_7"].min()
)

print(
    "rolling_mean_7 minimum:",
    df["rolling_mean_7"].min()
)

# ==================================================
# Price feature sanity
# ==================================================

print("\n====================================")
print("PRICE FEATURE SANITY")
print("====================================")

price_cols = [
    "price_change_1",
    "price_change_pct_1",
    "price_relative_7"
]

print(
    "Infinite values:"
)

for col in price_cols:
    print(
        f"{col}:",
        np.isinf(
            df[col].dropna()
        ).sum()
    )

# ==================================================
# Original raw columns preserved
# ==================================================

original_columns = [
    "id",
    "item_id",
    "dept_id",
    "cat_id",
    "store_id",
    "state_id",
    "d",
    "sales",
    "date",
    "wm_yr_wk",
    "weekday",
    "wday",
    "month",
    "year",
    "event_name_1",
    "event_type_1",
    "event_name_2",
    "event_type_2",
    "snap_CA",
    "snap_TX",
    "snap_WI",
    "sell_price",
    "price_available"
]

print("\n====================================")
print("ORIGINAL COLUMN CHECK")
print("====================================")

missing_original = [
    col for col in original_columns
    if col not in columns
]

print(
    "Missing original columns:",
    missing_original
)

print(
    "All original columns preserved:",
    len(missing_original) == 0
)

print("\n====================================")
print("VERIFICATION COMPLETE")
print("====================================")

FINAL FEATURE DATASET VERIFICATION

Rows:
58,327,370

Row groups:
61

Columns:
 1. id
 2. item_id
 3. dept_id
 4. cat_id
 5. store_id
 6. state_id
 7. d
 8. sales
 9. date
10. wm_yr_wk
11. weekday
12. wday
13. month
14. year
15. event_name_1
16. event_type_1
17. event_name_2
18. event_type_2
19. snap_CA
20. snap_TX
21. snap_WI
22. sell_price
23. price_available
24. day_of_month
25. week_of_year
26. day_of_year
27. quarter
28. is_weekend
29. lag_1
30. lag_7
31. lag_14
32. lag_28
33. rolling_mean_7
34. rolling_mean_28
35. rolling_std_7
36. rolling_std_28
37. price_change_1
38. price_change_pct_1
39. price_relative_7
40. is_event_day
41. event_count
42. snap_active

ENGINEERED FEATURE CHECK
day_of_month             : True
week_of_year             : True
day_of_year              : True
quarter                  : True
is_weekend               : True
lag_1                    : True
lag_7                    : True
lag_14                   : True
lag_28                   : True
rolling_mean_7 